# Comparing to the previous row: LAG, LEAD and moving averages

Practice notebook for the datafromzero SQL lesson **Comparing to the previous row: LAG, LEAD and moving averages**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Orders by month

In [ ]:
con.sql("""
WITH monthly AS (
  SELECT CAST(DATE_TRUNC('month', ordered_at) AS DATE) AS month, COUNT(*) AS orders
  FROM orders
  GROUP BY month
)
SELECT month, orders
FROM monthly
ORDER BY month
""").show()

### Step 2. The row above and the row below: LAG and LEAD

In [ ]:
con.sql("""
WITH monthly AS (
  SELECT CAST(DATE_TRUNC('month', ordered_at) AS DATE) AS month, COUNT(*) AS orders
  FROM orders
  GROUP BY month
)
SELECT month, orders,
       LAG(orders)  OVER (ORDER BY month) AS prev_orders,
       LEAD(orders) OVER (ORDER BY month) AS next_orders
FROM monthly
ORDER BY month
""").show()

### Step 3. Change and percent change

In [ ]:
con.sql("""
WITH monthly AS (
  SELECT CAST(DATE_TRUNC('month', ordered_at) AS DATE) AS month, COUNT(*) AS orders
  FROM orders
  GROUP BY month
),
with_prev AS (
  SELECT month, orders, LAG(orders) OVER (ORDER BY month) AS prev_orders
  FROM monthly
)
SELECT month, orders, prev_orders,
       orders - prev_orders AS change,
       ROUND((orders - prev_orders) * 100.0 / NULLIF(prev_orders, 0), 1) AS pct_change
FROM with_prev
ORDER BY month
""").show()

### Step 4. The same comparison inside each cafe

In [ ]:
con.sql("""
WITH monthly AS (
  SELECT store_id, CAST(DATE_TRUNC('month', ordered_at) AS DATE) AS month, COUNT(*) AS orders
  FROM orders
  GROUP BY store_id, month
),
with_prev AS (
  SELECT store_id, month, orders,
         LAG(orders) OVER (PARTITION BY store_id ORDER BY month) AS prev_orders
  FROM monthly
)
SELECT store_id, month, orders, prev_orders,
       ROUND((orders - prev_orders) * 100.0 / NULLIF(prev_orders, 0), 1) AS pct_change
FROM with_prev
ORDER BY store_id, month
""").show()

### Step 5. A 7-day moving average (predict the first rows of the series first)

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS day, COUNT(*) AS orders
  FROM orders
  GROUP BY day
),
smoothed AS (
  SELECT day, orders,
         ROUND(AVG(orders) OVER (ORDER BY day
                                 ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS avg_7_days
  FROM daily
)
SELECT day, orders, avg_7_days
FROM smoothed
WHERE day BETWEEN '2026-03-30' AND '2026-04-07'
ORDER BY day
""").show()

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS day, COUNT(*) AS orders
  FROM orders
  GROUP BY day
)
SELECT day, orders,
       ROUND(AVG(orders) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS avg_7_days
FROM daily
ORDER BY day
LIMIT 7
""").show()

### Step 6. The LAST_VALUE trap (predict the March row first)

In [ ]:
con.sql("""
WITH monthly AS (
  SELECT CAST(DATE_TRUNC('month', ordered_at) AS DATE) AS month, COUNT(*) AS orders
  FROM orders
  GROUP BY month
)
SELECT month, orders,
       LAST_VALUE(orders) OVER (ORDER BY month) AS last_default,
       LAST_VALUE(orders) OVER (ORDER BY month
                                ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_whole
FROM monthly
ORDER BY month
""").show()

### Step 7. Verify

In [ ]:
con.sql("""
WITH monthly AS (
  SELECT CAST(DATE_TRUNC('month', ordered_at) AS DATE) AS month, COUNT(*) AS orders
  FROM orders
  GROUP BY month
)
SELECT SUM(orders) AS total_orders
FROM monthly
""").show()

## Exercise 1. Compared with the same weekday last week

The `daily` stage is given, with a `compared` stage that does not compare anything yet. Add the orders of the same
weekday one week earlier and the change, for April 4 to April 10.

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS day, COUNT(*) AS orders
  FROM orders
  GROUP BY day
),
compared AS (
  SELECT day, orders
  FROM daily
)
SELECT day, orders
FROM compared
WHERE day BETWEEN '2026-04-04' AND '2026-04-10'
ORDER BY day
""").show()

## Exercise 2. A three-month average

Before you run anything, write one sentence in the first cell: which months will be averaged for January and for
June? Then smooth the monthly orders with an average of each month, the month before and the month after.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query:

## Exercise 3. Spot the flaw

A colleague wants the change from February to March, and the months after it. The query runs, but the first row is
not what was asked for. Find the flaw and write the fix.

In [ ]:
con.sql("""
WITH monthly AS (
  SELECT CAST(DATE_TRUNC('month', ordered_at) AS DATE) AS month, COUNT(*) AS orders
  FROM orders
  GROUP BY month
)
SELECT month, orders,
       orders - LAG(orders) OVER (ORDER BY month) AS change
FROM monthly
WHERE month >= '2026-03-01'
ORDER BY month
""").show()

In [ ]:
# Your corrected query: